# Vibrating string: eigenvalues of the Laplacian

A string fixed at both ends vibrates in standing waves, and each wave has its
own frequency. Discretize the string at n points and the waves are the
eigenvectors of the second-difference matrix T, tridiagonal with 2 on the
diagonal and −1 beside it. The frequencies are the square roots of its
eigenvalues. T is one of the few matrices whose eigenvalues are known in
closed form, which makes it a test of the eigensolver. Kronecker products of T
then give a membrane, and a small asymmetric change shows what the symmetric
solver can't handle.

**Uses:** `mtl5.eigh`, `mtl5.eigvalsh`, `mtl5.eigvals`, `mtl5.svdvals`,
`mtl5.is_symmetric`, `mtl5.generators.laplacian_1d`.

**From C++:** [`phase06_eigenvalue_svd/vibrating_string.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/phase06_eigenvalue_svd/vibrating_string.cpp).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import mtl5

# Chart styling: recessive axes and grid, ink for text, validated categorical hues.
SURFACE = "#fcfcfb"
INK, INK_2 = "#0b0b0b", "#52514e"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#d8d7d2",
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.color": "#e8e7e2",
        "grid.linewidth": 0.8,
        "text.color": INK,
        "xtick.color": INK_2,
        "ytick.color": INK_2,
        "font.size": 10,
        "figure.dpi": 120,
    }
)


def laplacian_1d(n):
    # MTL5's generator, tridiagonal [-1, 2, -1], as a dense NumPy array
    S = mtl5.generators.laplacian_1d(n)
    indptr, cols, vals = S.to_csr_arrays()
    D = np.zeros(S.shape)
    D[np.repeat(np.arange(S.shape[0]), np.diff(indptr)), cols] = vals
    return D

## The string: computed against exact

For the n × n matrix T, the k-th eigenvalue is λₖ = 4 sin²(kπ / (2(n + 1))),
and its eigenvector samples sin(kπx) at the points xⱼ = j / (n + 1). The
tutorial uses n = 8. `mtl5.eigh` returns the eigenvalues in ascending order
with the eigenvectors as columns.

In [ ]:
n = 8
T = laplacian_1d(n)
lam, Q = mtl5.eigh(mtl5.matrix(T))

k = np.arange(1, n + 1)
exact = 4.0 * np.sin(k * np.pi / (2.0 * (n + 1))) ** 2
for kk, computed, analytic in zip(k, lam, exact):
    error = abs(computed - analytic)
    print(f"k = {kk}: computed {computed:.12f}  exact {analytic:.12f}  error {error:.1e}")

assert np.allclose(lam, exact, rtol=0, atol=1e-12)
assert np.allclose(Q.T @ Q, np.eye(n), atol=1e-12)  # orthonormal modes

Each eigenvector is the sampled sine, up to its sign and length. Plotted
against the continuous mode, the low modes are smooth. The highest one
alternates from point to point, the shortest wave 8 points can carry.

In [ ]:
j = np.arange(1, n + 1)
x_pts = j / (n + 1)
x_fine = np.linspace(0.0, 1.0, 400)

fig, axes = plt.subplots(2, 2, figsize=(7.0, 4.6), sharex=True, sharey=True)
for ax, kk in zip(axes.flat, [1, 2, 3, 8]):
    mode = np.sin(kk * np.pi * x_pts)
    mode /= np.linalg.norm(mode)
    v = Q[:, kk - 1] * np.sign(Q[:, kk - 1] @ mode)  # fix the eigenvector's sign
    assert np.allclose(v, mode, atol=1e-10), kk

    scale = np.linalg.norm(np.sin(kk * np.pi * x_pts))
    ax.plot(x_fine, np.sin(kk * np.pi * x_fine) / scale, color=SERIES[0], linewidth=1)
    ax.plot(np.r_[0, x_pts, 1], np.r_[0, v, 0], "o-", color=SERIES[1], markersize=4)
    ax.set_title(f"mode {kk}: λ = {lam[kk - 1]:.4f}", fontsize=9)
    ax.axhline(0.0, color=INK_2, linewidth=0.6)
fig.suptitle("Eigenvectors of T (dots) on the continuous modes sin(kπx)")
plt.tight_layout()
plt.show()

## A membrane from two strings

The Laplacian of an n × n grid is L = I ⊗ T + T ⊗ I: T acts along one axis
and the identity along the other. NumPy's `np.kron` builds it. The
eigenvectors of L are products of string modes, one per axis, so each
eigenvalue of L is a sum λᵢ + λⱼ of two eigenvalues of T. The tutorial checks
that for n = 4, where L is 16 × 16.

In [ ]:
n2 = 4
T2 = laplacian_1d(n2)
I2 = np.eye(n2)
L = np.kron(I2, T2) + np.kron(T2, I2)

lam_2d = mtl5.eigvalsh(mtl5.matrix(L))
lam_1d = mtl5.eigvalsh(mtl5.matrix(T2))
pair_sums = np.sort(np.add.outer(lam_1d, lam_1d).ravel())

print("2D eigenvalues:", np.round(lam_2d, 6))
print("max |λ_2D − (λ_i + λ_j)| =", f"{np.abs(lam_2d - pair_sums).max():.1e}")
assert np.allclose(lam_2d, pair_sums, rtol=0, atol=1e-12)

On a finer grid the product structure is visible in the modes themselves. The
lowest mode of L is the fundamental of the string in both directions,
sin(πx) sin(πy). Mode (2, 2) has a node line along each axis. Both are single
eigenvalues; the modes (1, 2) and (2, 1) share one, and their eigenvectors
can mix, so they aren't plotted.

In [ ]:
g = 16
Tg = laplacian_1d(g)
Lg = np.kron(np.eye(g), Tg) + np.kron(Tg, np.eye(g))
lam_g, Q_g = mtl5.eigh(mtl5.matrix(Lg))
# rows: the string's modes 1, 2 and 3 on g points
s = np.sin(np.outer(np.arange(1, 4), np.arange(1, g + 1)) * np.pi / (g + 1))
lam_s = 4.0 * np.sin(np.arange(1, 4) * np.pi / (2.0 * (g + 1))) ** 2

fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.4))
for ax, (a, b) in zip(axes, [(1, 1), (2, 2)]):
    idx = int(np.argmin(np.abs(lam_g - (lam_s[a - 1] + lam_s[b - 1]))))
    product = np.kron(s[b - 1], s[a - 1])
    product /= np.linalg.norm(product)
    v = Q_g[:, idx] * np.sign(Q_g[:, idx] @ product)
    assert np.allclose(v, product, atol=1e-8), (a, b)

    im = ax.imshow(v.reshape(g, g), cmap="RdBu_r", vmin=-0.13, vmax=0.13, origin="lower")
    ax.set_title(f"mode ({a}, {b}): λ = {lam_g[idx]:.4f}", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(im, ax=axes, shrink=0.8)
fig.suptitle(f"Two eigenvectors of the Laplacian of a {g} × {g} grid ({g * g} unknowns)")
plt.show()

## An asymmetric change: complex eigenvalues

A symmetric matrix has real eigenvalues, and `eigh` and `eigvalsh` assume
that. Change one corner entry of T, B[0, n − 1] = s, and B is no longer
symmetric. `mtl5.eigvals`, the general solver, returns complex eigenvalues.

The C++ example sets s = 0.5 and expects complex ones, but at that size all
four stay real (stillwater-sc/mtl5#938). Sweeping s shows when they leave the
real axis: at s = 5/4 the eigenvalues meet in two pairs, and continue as
complex conjugate pairs. Through all of it their sum is the trace of B, 8.

In [ ]:
assert not mtl5.is_symmetric(mtl5.matrix(T2 + 0.5 * np.eye(n2, k=n2 - 1)))

sweep = np.linspace(0.0, 2.5, 251)
paths = []
for corner in sweep:
    B = T2.copy()
    B[0, n2 - 1] = corner
    w = mtl5.eigvals(mtl5.matrix(B))
    assert np.isclose(w.sum().real, np.trace(B)) and abs(w.sum().imag) < 1e-10
    # B is real, so its eigenvalues come in conjugate pairs
    assert np.allclose(np.sort_complex(w), np.sort_complex(w.conj()), atol=1e-6)
    paths.append(w)
paths = np.array(paths)

fig, ax = plt.subplots(figsize=(5.8, 3.6))
sc = ax.scatter(paths.real.ravel(), paths.imag.ravel(), c=np.repeat(sweep, n2), cmap="viridis", s=5)
fig.colorbar(sc, ax=ax, label="corner entry s")
ax.plot(lam_1d, np.zeros(n2), "o", color=SERIES[1], label="s = 0: eigenvalues of T")
ax.set_xlabel("Re λ")
ax.set_ylabel("Im λ")
ax.set_title("Eigenvalues of T with B[0, 3] = s, for s from 0 to 2.5")
ax.legend(frameon=False, loc="upper center")
ax.grid(True)
plt.tight_layout()
plt.show()

at = {c: mtl5.eigvals(mtl5.matrix(T2 + c * np.eye(n2, k=n2 - 1))) for c in (0.5, 2.0)}
print("s = 0.5:", np.round(at[0.5], 4))
print("s = 2.0:", np.round(at[2.0], 4))
assert np.allclose(at[0.5].imag, 0.0)  # still real at the C++ example's 0.5
assert (np.abs(at[2.0].imag) > 0.1).sum() == 4  # two conjugate pairs at 2

## Why exactly 5/4

The boundary is exact. Expand det(B − λI) along the first row: the corner
entry s is multiplied by its cofactor, (−1)⁰⁺³ times the product of the three
−1 entries below the diagonal, which is 1. So the corner adds a constant to
the characteristic polynomial,

det(B − λI) = det(T − λI) + s.

With μ = λ − 2, det(T − λI) = μ⁴ − 3μ² + 1. That polynomial is even in μ, and
both of its local minima, at μ² = 3/2, have the value −5/4. Adding s lifts the
whole curve: at s = 5/4 both minima touch zero at once, giving two double
eigenvalues λ = 2 ± √(3/2), and beyond it the four roots are complex. Lowering
it by more than 1 pulls the central maximum, μ = 0, below zero instead. So the
eigenvalues are real exactly for −1 ≤ s ≤ 5/4.

In [ ]:
def charpoly_gap(s, lam):
    B = T2 + s * np.eye(n2, k=n2 - 1)
    return np.linalg.det(B - lam * np.eye(n2)) - np.linalg.det(T2 - lam * np.eye(n2))


for s_ in (0.5, 1.25, 2.0):
    for lam_ in (-1.0, 0.3, 2.0, 3.7):
        assert np.isclose(charpoly_gap(s_, lam_), s_, rtol=0, atol=1e-12)


def max_imag(s):
    return np.abs(mtl5.eigvals(mtl5.matrix(T2 + s * np.eye(n2, k=n2 - 1))).imag).max()


s_axis = np.linspace(-2.0, 2.5, 451)
fig, ax = plt.subplots(figsize=(5.8, 3.0))
ax.plot(s_axis, [max_imag(s) for s in s_axis], color=SERIES[0])
for edge in (-1.0, 1.25):
    ax.axvline(edge, color=SERIES[1], linestyle="--", linewidth=1)
ax.set_xlabel("corner entry s")
ax.set_ylabel("max |Im λ|")
ax.set_title("Real for −1 ≤ s ≤ 5/4, complex outside", fontsize=10)
ax.grid(True)
plt.tight_layout()
plt.show()

# on either side of each edge (at the edge itself the eigenvalues are double)
assert max_imag(1.2) < 1e-8 and max_imag(-0.9) < 1e-8
assert max_imag(1.3) > 0.05 and max_imag(-1.2) > 0.05
double = np.sort(mtl5.eigvals(mtl5.matrix(T2 + 1.25 * np.eye(n2, k=n2 - 1))).real)
print(
    "s = 5/4:", np.round(double, 5), " 2 ± sqrt(3/2) =", np.round([2 - 1.5**0.5, 2 + 1.5**0.5], 5)
)
assert np.allclose(double, np.repeat([2 - 1.5**0.5, 2 + 1.5**0.5], 2), atol=1e-6)

## Singular values and eigenvalues

For a symmetric matrix the singular values are the absolute values of the
eigenvalues. For a matrix that isn't symmetric, they differ. B with s = 2 has
complex eigenvalues but, like every matrix, real nonnegative singular values.

In [ ]:
sv = np.sort(mtl5.svdvals(mtl5.matrix(T2)))
abs_eig = np.sort(np.abs(lam_1d))
print("T:  |eigenvalues|", np.round(abs_eig, 8), " singular values", np.round(sv, 8))
assert np.allclose(sv, abs_eig, rtol=0, atol=1e-12)

B2 = T2 + 2.0 * np.eye(n2, k=n2 - 1)
sv_b = np.sort(mtl5.svdvals(mtl5.matrix(B2)))
abs_eig_b = np.sort(np.abs(at[2.0]))
print("B:  |eigenvalues|", np.round(abs_eig_b, 4), " singular values", np.round(sv_b, 4))
assert not np.allclose(sv_b, abs_eig_b, atol=1e-3)

## From C++

| MTL5 (C++) | mtl5 (Python) |
| --- | --- |
| `eigenvalue_symmetric(T)` | `mtl5.eigvalsh(T)`, or `mtl5.eigh(T)` for the eigenvectors too |
| `eigenvalue(B)` | `mtl5.eigvals(B)`, or `mtl5.eig(B)` for the eigenvectors too |
| `svd(T, U, S, V)`, with S a diagonal matrix | `U, s, V = mtl5.svd(T)`, with s a vector; `mtl5.svdvals(T)` for s alone |
| `kron(I, T)` with `identity2D<double> I(n)` | `np.kron(np.eye(n), T)` |
| a sorted vector of pairs λᵢ + λⱼ | `np.sort(np.add.outer(lam, lam).ravel())` |

`eigh` and `eigvalsh` return the eigenvalues in ascending order, as
`eigenvalue_symmetric` does; `eigvals` returns them in no particular order.